# CEMADEN — agregação por estação

## Objetivo
Ler os arquivos `data_*.csv` da pasta `Cemaden` e agregar `valorMedida` por estação em três granularidades:

- dia;
- semana (segunda a domingo);
- mês.

Cada linha agregada representa **uma estação e um período**. Meses diferentes nunca são somados na mesma linha.

Para cada estação e período, o notebook calcula:

- `soma_mm`: soma das leituras;
- `media_mm`: média das leituras;
- `n_obs`: quantidade de leituras;
- `inicio` e `fim`: primeira e última leitura do período.

## Premissas
- `valorMedida` está em mm e é mantido nessa unidade; nenhuma conversão é feita.
- A soma é a soma das leituras observadas, não uma estimativa para lacunas.
- Lacunas não são preenchidas com zero: período sem leitura não aparece nas tabelas de dia/semana.
- A semana começa na segunda-feira e termina no domingo.
- A primeira e a última semana podem ser parciais; confira `n_obs` antes de comparar.
- `resumo` e `por_mes` têm uma linha por estação e mês, inclusive para estações sem leitura.
- Cada hora com leitura é classificada como `com_chuva` (alguma leitura > 0) ou `sem_chuva` (todas as leituras são 0).
- Estação/mês sem nenhuma leitura é classificado como `sem_leitura`.
- `pct_horas_com_chuva` e `pct_horas_sem_chuva` usam apenas as horas com leitura; não há premissa de horas esperadas.
- `resumo_mensal` tem uma linha por mês e mostra a contagem e o percentual de estações em cada situação.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PASTA = Path("Cemaden")
ARQUIVO = "data_OUT.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [ ]:
arquivos = sorted(PASTA.glob(ARQUIVO))
print(f"Pasta: {PASTA.resolve()}")
print(f"{len(arquivos)} arquivo(s) encontrado(s):")
for arquivo in arquivos:
    print(f"- {arquivo.name} ({arquivo.stat().st_size / 1_000_000:.2f} MB)")

In [ ]:
COLS_OBRIG = [
    "municipio",
    "codEstacao",
    "uf",
    "nomeEstacao",
    "latitude",
    "longitude",
    "datahora",
    "valorMedida",
]

DTYPES = {
    "municipio": "string",
    "codEstacao": "string",
    "uf": "string",
    "nomeEstacao": "string",
    "latitude": "float64",
    "longitude": "float64",
    "valorMedida": "float64",
}


def ler_cemaden(caminho: str | Path) -> pd.DataFrame:
    caminho = Path(caminho)
    if not caminho.is_absolute():
        caminho = PASTA / caminho

    arquivos = sorted(caminho.parent.glob(caminho.name))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para: {caminho}")

    partes = []
    for arq in arquivos:
        parte = pd.read_csv(
            arq,
            sep=";",
            encoding="utf-8-sig",
            decimal=",",
            dtype=DTYPES,
            parse_dates=["datahora"],
        )
        falta = set(COLS_OBRIG) - set(parte.columns)
        if falta:
            raise ValueError(f"{arq.name}: colunas ausentes: {sorted(falta)}")

        parte["arquivo_origem"] = arq.name
        partes.append(parte)

    df = pd.concat(partes, ignore_index=True)

    for coluna in ["municipio", "codEstacao", "uf", "nomeEstacao"]:
        df[coluna] = df[coluna].str.strip()

    df["datahora"] = pd.to_datetime(df["datahora"], errors="raise")
    return df.sort_values(["codEstacao", "datahora"]).reset_index(drop=True)

In [ ]:
df_cemaden = ler_cemaden(ARQUIVO)
print(f"Linhas: {len(df_cemaden):,} | Estações: {df_cemaden['codEstacao'].nunique():,}")
print(f"Período: {df_cemaden['datahora'].min()} -> {df_cemaden['datahora'].max()}")
display(df_cemaden.head())

In [ ]:
def validar(df: pd.DataFrame) -> None:
    print(f"Linhas: {len(df):,}")
    print(f"Estações: {df['codEstacao'].nunique():,}")
    print(f"Arquivos: {df['arquivo_origem'].nunique():,}")
    print(f"Período: {df['datahora'].min()} -> {df['datahora'].max()}")

    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    if nulos.empty:
        print("Nulos: nenhum")
    else:
        print("Nulos por coluna:")
        print(nulos.to_string())

    dups = int(df.duplicated(["codEstacao", "datahora"]).sum())
    print(f"Duplicadas por estação + datahora: {dups:,}")
    if dups:
        raise ValueError("Existem leituras duplicadas para a mesma estação e datahora.")

    meta = df.groupby("codEstacao", dropna=False).agg(
        nomes=("nomeEstacao", "nunique"),
        municipios=("municipio", "nunique"),
    )
    incons = meta[(meta["nomes"] > 1) | (meta["municipios"] > 1)]
    print(f"Estações com metadados inconsistentes: {len(incons):,}")

    print(
        "valorMedida: "
        f"min={df['valorMedida'].min():.2f}, "
        f"max={df['valorMedida'].max():.2f}, "
        f"media={df['valorMedida'].mean():.4f}, "
        f"zeros={(df['valorMedida'] == 0).sum():,}, "
        f"negativos={(df['valorMedida'] < 0).sum():,}"
    )

    leituras = df.groupby("codEstacao", dropna=False).size()
    print("Leituras por estação:")
    print(leituras.describe().to_string())


validar(df_cemaden)

In [ ]:
COLS_ESTACAO = [
    "codEstacao",
    "nomeEstacao",
    "municipio",
    "uf",
    "latitude",
    "longitude",
]

PERIODOS = {
    "dia": lambda serie: serie.dt.normalize(),
    "semana": lambda serie: serie.dt.to_period("W-SUN").dt.start_time,
    "mes": lambda serie: serie.dt.to_period("M").dt.start_time,
}


def agregar(df: pd.DataFrame, granularidade: str) -> pd.DataFrame:
    if granularidade not in PERIODOS:
        raise ValueError(f"Granularidade inválida: {granularidade!r}")

    periodo = PERIODOS[granularidade](df["datahora"])
    tabela = (
        df.assign(periodo=periodo)
        .groupby(["codEstacao", "periodo"], as_index=False, dropna=False)
        .agg(
            nomeEstacao=("nomeEstacao", "first"),
            municipio=("municipio", "first"),
            uf=("uf", "first"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
            inicio=("datahora", "min"),
            fim=("datahora", "max"),
            n_obs=("valorMedida", "count"),
            soma_mm=("valorMedida", "sum"),
            media_mm=("valorMedida", "mean"),
        )
        .sort_values(["codEstacao", "periodo"])
        .reset_index(drop=True)
    )
    return tabela[
        COLS_ESTACAO + ["periodo", "inicio", "fim", "n_obs", "soma_mm", "media_mm"]
    ]


def resumo_mes(df: pd.DataFrame) -> pd.DataFrame:
    base = df.assign(periodo=PERIODOS["mes"](df["datahora"]))
    tabela = (
        base.groupby("periodo", as_index=False, dropna=False)
        .agg(
            inicio=("datahora", "min"),
            fim=("datahora", "max"),
            n_obs=("valorMedida", "count"),
            n_estacoes_com_leitura=("codEstacao", "nunique"),
            soma_mm=("valorMedida", "sum"),
            media_mm=("valorMedida", "mean"),
        )
        .sort_values("periodo")
        .reset_index(drop=True)
    )
    return tabela[["periodo", "inicio", "fim", "n_obs", "n_estacoes_com_leitura", "soma_mm", "media_mm"]]


def add_horas(resumo: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    base = df.assign(
        periodo=PERIODOS["mes"](df["datahora"]),
        hora=df["datahora"].dt.floor("h"),
    )

    horas = (
        base.groupby(["codEstacao", "periodo", "hora"], as_index=False)
        .agg(chuva=("valorMedida", lambda s: bool((s > 0).any())))
    )

    obs = (
        horas.groupby(["codEstacao", "periodo"], as_index=False)
        .agg(
            horas_com_leitura=("hora", "nunique"),
            horas_com_chuva=("chuva", "sum"),
        )
    )

    obs["horas_sem_chuva"] = (
        obs["horas_com_leitura"] - obs["horas_com_chuva"]
    )
    obs["pct_horas_com_chuva"] = (
        obs["horas_com_chuva"] / obs["horas_com_leitura"] * 100
    ).round(2)
    obs["pct_horas_sem_chuva"] = (100 - obs["pct_horas_com_chuva"]).round(2)

    return resumo.merge(
        obs,
        on=["codEstacao", "periodo"],
        how="left",
        validate="one_to_one",
    )


def grade_estacao_mes(resumo: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    estacoes = df[
        [
            "codEstacao",
            "nomeEstacao",
            "municipio",
            "uf",
            "latitude",
            "longitude",
        ]
    ].drop_duplicates("codEstacao")

    periodos = pd.DataFrame(
        {"periodo": sorted(df["datahora"].dt.to_period("M").dt.start_time.unique())}
    )

    grade = estacoes.merge(periodos, how="cross")
    dados = resumo.drop(
        columns=["nomeEstacao", "municipio", "uf", "latitude", "longitude"]
    )
    completo = grade.merge(dados, on=["codEstacao", "periodo"], how="left")

    for coluna in ["n_obs", "horas_com_leitura", "horas_com_chuva", "horas_sem_chuva"]:
        completo[coluna] = completo[coluna].fillna(0).astype(int)

    completo["soma_mm"] = completo["soma_mm"].fillna(0.0)
    completo["situacao"] = np.select(
        [
            completo["horas_com_leitura"].eq(0),
            completo["horas_com_chuva"].gt(0),
        ],
        ["sem_leitura", "com_chuva"],
        default="sem_chuva",
    )

    return completo.sort_values(["codEstacao", "periodo"]).reset_index(drop=True)


def resumo_situacao(resumo: pd.DataFrame) -> pd.DataFrame:
    tabela = (
        resumo.groupby(["periodo", "situacao"], dropna=False)
        .size()
        .unstack(fill_value=0)
    )

    for coluna in ["com_chuva", "sem_chuva", "sem_leitura"]:
        if coluna not in tabela.columns:
            tabela[coluna] = 0

    tabela = tabela[["com_chuva", "sem_chuva", "sem_leitura"]]
    tabela.columns = ["n_st_monit_ch", "n_st_monit_ch_z", "n_st_sem_leitura"]
    tabela["n_st_total"] = tabela.sum(axis=1)

    for de, para in [
        ("n_st_monit_ch", "pct_st_monit_ch"),
        ("n_st_monit_ch_z", "pct_st_monit_ch_z"),
        ("n_st_sem_leitura", "pct_st_sem_leitura"),
    ]:
        tabela[para] = (tabela[de] / tabela["n_st_total"] * 100).round(2)

    return tabela.reset_index()

In [ ]:
resumo = agregar(df_cemaden, "mes")
resumo = add_horas(resumo, df_cemaden)
resumo = grade_estacao_mes(resumo, df_cemaden)

if resumo.duplicated(["codEstacao", "periodo"]).any():
    raise ValueError("O resumo tem mais de uma linha para a mesma estação e mês.")

print(
    f"Linhas no resumo (estação × mês): {len(resumo):,} | "
    f"Estações: {resumo['codEstacao'].nunique():,} | "
    f"Meses: {resumo['periodo'].nunique():,}"
)

cols = [
    "codEstacao",
    "periodo",
    "n_obs",
    "soma_mm",
    "horas_com_leitura",
    "horas_com_chuva",
    "horas_sem_chuva",
    "pct_horas_com_chuva",
    "pct_horas_sem_chuva",
    "situacao",
]
display(resumo[cols].head(10))

print("Estações/meses sem leitura:")
display(
    resumo.loc[
        resumo["situacao"].eq("sem_leitura"),
        ["codEstacao", "nomeEstacao", "periodo", "situacao"],
    ]
)

In [ ]:
por_dia = agregar(df_cemaden, "dia")
print(
    f"Linhas: {len(por_dia):,} | "
    f"Estações: {por_dia['codEstacao'].nunique():,} | "
    f"Dias: {por_dia['periodo'].nunique():,}"
)
display(por_dia.head(10))

In [ ]:
por_semana = agregar(df_cemaden, "semana")
print(
    f"Linhas: {len(por_semana):,} | "
    f"Estações: {por_semana['codEstacao'].nunique():,} | "
    f"Semanas: {por_semana['periodo'].nunique():,}"
)
display(por_semana.head(10))

In [ ]:
por_mes = resumo.copy()
print(
    f"Linhas: {len(por_mes):,} | "
    f"Estações: {por_mes['codEstacao'].nunique():,} | "
    f"Meses: {por_mes['periodo'].nunique():,}"
)
display(por_mes.head(10))

resumo_mensal = resumo_mes(df_cemaden)
situacao_mensal = resumo_situacao(resumo)
resumo_mensal = resumo_mensal.merge(situacao_mensal, on="periodo", how="left")

print("Resumo geral por mês:")
display(resumo_mensal)

In [ ]:
def conferir(tabela: pd.DataFrame, df: pd.DataFrame, nome: str) -> None:
    chave = ["periodo"] if "codEstacao" not in tabela.columns else ["codEstacao", "periodo"]
    dups = int(tabela.duplicated(chave).sum())
    if dups:
        raise ValueError(f"{nome}: chave {chave} duplicada.")

    if not np.isclose(tabela["soma_mm"].sum(), df["valorMedida"].sum()):
        raise ValueError(f"{nome}: a soma agregada não fecha com a base.")

    if tabela["n_obs"].sum() != df["valorMedida"].count():
        raise ValueError(f"{nome}: a contagem de observações não fecha com a base.")

    print(f"{nome}: OK")


conferir(resumo, df_cemaden, "resumo (estação × mês)")
conferir(por_dia, df_cemaden, "dia")
conferir(por_semana, df_cemaden, "semana")
conferir(por_mes, df_cemaden, "mês")
conferir(resumo_mensal, df_cemaden, "geral por mês")

## Como usar com outro arquivo

- `ARQUIVO = "data_*.csv"` lê todos os arquivos da pasta.
- Para analisar apenas um mês, altere para `ARQUIVO = "data_JAN.csv"` e execute novamente as células.
- `resumo` e `por_mes` têm uma linha por estação e mês; estações sem leitura aparecem como `sem_leitura`.
- `com_chuva`: a estação/mês tem pelo menos uma leitura maior que zero.
- `sem_chuva`: a estação/mês tem leituras, mas todas são zero.
- `sem_leitura`: a estação não tem nenhuma leitura naquele mês.
- `pct_horas_com_chuva` e `pct_horas_sem_chuva` são calculados apenas sobre as horas com leitura.
- `resumo_mensal` mostra quantas estações e qual percentual estão em cada situação (`n_st_monit_ch`, `n_st_monit_ch_z`, `n_st_sem_leitura` e seus `pct_*`).
- `soma_mm` não estima lacunas e não converte taxa em acumulado; a interpretação depende da unidade e do intervalo de `valorMedida` na fonte.


In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text


def ler_env(caminho: str | Path = ".env") -> None:
    caminho = Path(caminho)
    if not caminho.exists():
        return
    for linha in caminho.read_text(encoding="utf-8").splitlines():
        linha = linha.strip()
        if not linha or linha.startswith("#") or "=" not in linha:
            continue
        chave, valor = linha.split("=", 1)
        os.environ.setdefault(chave.strip(), valor.strip().strip('"').strip("'"))


ler_env()

local_db_url = os.getenv("LOCAL_DB_URL")
if not local_db_url:
    raise RuntimeError("LOCAL_DB_URL não está definida. Confira se Notebookes/.env existe.")
engine_local = create_engine(local_db_url)

supabase_db_url = os.getenv("SUPABASE_DB_URL")
if not supabase_db_url:
    raise RuntimeError(
        "SUPABASE_DB_URL não está definida. "
        "Confira se Notebookes/.env existe e rode a célula de setup antes desta."
    )
engine_claud = create_engine(supabase_db_url)


def persistir(df: pd.DataFrame, nome: str, con, index_label: str = "index") -> None:
    df = df.copy()
    
    if "media_mm" in df.columns:
        df["media_mm"] = df["media_mm"].round(2)
    
    for periodo in df["periodo"].unique():
        try:
            n = pd.read_sql(
                text(f'SELECT count(*) AS n FROM "{nome}" WHERE periodo = :p'),
                con,
                params={"p": periodo},
            )["n"][0]
        except Exception:
            n = 0

        if n > 0:
            print(f"{nome:<22} {periodo} já existe — pulando")
            continue

        try:
            max_idx = pd.read_sql(
                text(f'SELECT COALESCE(MAX("{index_label}"), 0) AS m FROM "{nome}"'),
                con,
            )["m"][0]
        except Exception:
            max_idx = 0

        parte = df[df["periodo"] == periodo].copy()
        parte.index = range(int(max_idx) + 1, int(max_idx) + 1 + len(parte))
        parte.to_sql(
            name=nome,
            con=con,
            if_exists="append",
            index=True,
            index_label=index_label,
            method="multi",
        )
        print(f"{nome:<22} {periodo} inserido ({len(parte)} linhas)")

In [ ]:
# local
persistir(resumo, "df_cmd_mensal", engine_local)
persistir(resumo_mensal, "df_cmd_mgeral", engine_local)

# supabase
persistir(resumo, "df_cmd_mensal", engine_claud)
persistir(resumo_mensal, "df_cmd_mgeral", engine_claud)